# 02 — Diseño de features

Este notebook convierte hallazgos de `01_eda.ipynb` en variables que también existen al momento de predecir. La transformación está implementada en `src/features.py` para que los siguientes experimentos importen la misma lógica, en lugar de copiarla.

## Regla contra fuga de datos

`Exited` solo se utiliza en este notebook para describir diferencias observadas en train. `ChurnFeatureEngineer.fit` y `.transform` no leen la etiqueta, no calculan tasas de churn y no ajustan estadísticos con todos los datos. Los cortes de edad y balance son reglas fijas derivadas de la EDA; ambos valores fuente están disponibles en la OBT antes de que ocurra o se prediga churn.

La evidencia de asociación no demuestra que las features mejoren el modelo. Esa hipótesis deberá evaluarse en la misma validación cruzada de los modelos candidatos.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "datos.py").exists():
    raise FileNotFoundError("Abre este notebook desde la carpeta PSet3.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.datos import load_obt
from src.features import ChurnFeatureEngineer

pd.set_option("display.max_columns", 50)

In [2]:
obt = load_obt()
obt.columns = obt.columns.str.lower()
train = obt.loc[obt["source_split"].astype("string").str.lower().eq("train")].copy()
competition_test = obt.loc[
    obt["source_split"].astype("string").str.lower().eq("test")
].copy()

if train.empty or train["exited"].isna().any():
    raise ValueError("Se requiere train con etiquetas para evaluar la evidencia de EDA.")
if competition_test["exited"].notna().any():
    raise ValueError("El split test debe permanecer sin etiqueta.")

train["age"] = pd.to_numeric(train["age"], errors="coerce")
train["balance"] = pd.to_numeric(train["balance"], errors="coerce")
train["exited"] = pd.to_numeric(train["exited"], errors="coerce")
if train[["age", "balance", "exited"]].isna().any().any():
    raise ValueError("Hay faltantes en age, balance o exited; revisar la calidad antes de calcular evidencia.")

print(f"Train etiquetado para análisis: {len(train):,} filas")
print(f"Test sin etiqueta para comprobar disponibilidad: {len(competition_test):,} filas")

Train etiquetado para análisis: 15,000 filas
Test sin etiqueta para comprobar disponibilidad: 10,000 filas


## Evidencia y decisión

La siguiente tabla recalcula la evidencia en el snapshot cargado. La variable de edad resume el cambio marcado que aparece entre los grupos 30–39 y 40–49 de la EDA. La variable de balance distingue la masa en cero del resto. Se reportan tamaños de grupo junto a las tasas para contextualizar cada proporción.

In [3]:
age_rate = (
    train.assign(age_40_plus=train["age"].ge(40))
    .groupby("age_40_plus", observed=True)["exited"]
    .agg(filas="size", tasa_churn="mean")
)
age_rate.index = age_rate.index.map({False: "menor_de_40", True: "40_o_mas"})
age_rate["tasa_churn_pct"] = (100 * age_rate["tasa_churn"]).round(2)

balance_rate = (
    train.assign(balance_status=train["balance"].gt(0).map({False: "saldo_cero", True: "saldo_positivo"}))
    .groupby("balance_status", observed=True)["exited"]
    .agg(filas="size", tasa_churn="mean")
)
balance_rate["tasa_churn_pct"] = (100 * balance_rate["tasa_churn"]).round(2)

display(age_rate[["filas", "tasa_churn_pct"]])
display(balance_rate[["filas", "tasa_churn_pct"]])

feature_decisions = pd.DataFrame([
    {
        "feature_derivada": "age_40_plus",
        "regla": "40_o_mas si 40 <= age <= 100; menor_de_40 si 18 <= age < 40.",
        "evidencia_train": (
            f"Churn: {age_rate.loc['40_o_mas', 'tasa_churn_pct']:.2f}% en 40+ "
            f"(n={int(age_rate.loc['40_o_mas', 'filas']):,}) frente a "
            f"{age_rate.loc['menor_de_40', 'tasa_churn_pct']:.2f}% en <40 "
            f"(n={int(age_rate.loc['menor_de_40', 'filas']):,})."
        ),
        "disponible_en_inferencia": "Sí: age existe en la OBT de train y test.",
        "codificacion": "Categórica; OneHotEncoder(handle_unknown='ignore') dentro del pipeline de sklearn.",
    },
    {
        "feature_derivada": "balance_status",
        "regla": "saldo_cero si balance == 0; saldo_positivo si balance > 0.",
        "evidencia_train": (
            f"{100 * train['balance'].eq(0).mean():.2f}% tiene saldo cero; churn: "
            f"{balance_rate.loc['saldo_positivo', 'tasa_churn_pct']:.2f}% con saldo positivo "
            f"(n={int(balance_rate.loc['saldo_positivo', 'filas']):,}) frente a "
            f"{balance_rate.loc['saldo_cero', 'tasa_churn_pct']:.2f}% con saldo cero "
            f"(n={int(balance_rate.loc['saldo_cero', 'filas']):,})."
        ),
        "disponible_en_inferencia": "Sí: balance existe en la OBT de train y test.",
        "codificacion": "Categórica; OneHotEncoder(handle_unknown='ignore') dentro del pipeline de sklearn.",
    },
])
display(feature_decisions)

,filas,tasa_churn_pct
age_40_plus,,
menor_de_40,9682,7.37
40_o_mas,5318,44.79


,filas,tasa_churn_pct
balance_status,,
saldo_cero,9714,16.18
saldo_positivo,5286,28.83


,feature_derivada,regla,evidencia_train,disponible_en_inferencia,codificacion
0,age_40_plus,40_o_mas si 40 <= age <= 100; menor_de_40 si 1...,"Churn: 44.79% en 40+ (n=5,318) frente a 7.37% ...",Sí: age existe en la OBT de train y test.,Categórica; OneHotEncoder(handle_unknown='igno...
1,balance_status,saldo_cero si balance == 0; saldo_positivo si ...,64.76% tiene saldo cero; churn: 28.83% con sal...,Sí: balance existe en la OBT de train y test.,Categórica; OneHotEncoder(handle_unknown='igno...


## Aplicar la transformación

Se excluyen target e identificadores del conjunto de features. La banda de edad precomputada por el PSet 2 también se elimina del resultado para no duplicar la representación; `age_40_plus` se deriva directamente de `age`. El codificador se añadirá junto con imputación y escalado dentro de `src/pipeline.py` en el siguiente paso.

In [4]:
excluded_from_model = [
    "exited", "observation_id", "customer_id", "customer_id_raw", "surname",
    "source_split",
]
X_train = train.drop(columns=excluded_from_model, errors="ignore")
X_competition_test = competition_test.drop(columns=excluded_from_model, errors="ignore")

feature_engineer = ChurnFeatureEngineer()
X_train_features = feature_engineer.fit_transform(X_train)
X_test_features = feature_engineer.transform(X_competition_test)

assert "exited" not in X_train_features.columns
assert "exited" not in X_test_features.columns
assert "age_band" not in X_train_features.columns
assert "age_40_plus" in X_train_features.columns
assert "balance_status" in X_train_features.columns
assert X_train_features.columns.tolist() == X_test_features.columns.tolist()

print("Columnas de entrada:", X_train.shape[1])
print("Columnas después de derivar y quitar la banda duplicada:", X_train_features.shape[1])
print("Target usado para crear features: no")
display(X_train_features[["age", "age_40_plus", "balance", "balance_status"]].head())
display(X_train_features[["age_40_plus", "balance_status"]].nunique().rename("categorias").to_frame())
print("Nombres para el siguiente ColumnTransformer:", feature_engineer.get_feature_names_out().tolist())

Columnas de entrada: 13
Columnas después de derivar y quitar la banda duplicada: 12
Target usado para crear features: no


,age,age_40_plus,balance,balance_status
0,42.0,40_o_mas,0.00,saldo_cero
1,29.0,menor_de_40,0.00,saldo_cero
2,37.0,menor_de_40,0.00,saldo_cero
3,44.0,40_o_mas,141078.37,saldo_positivo
4,31.0,menor_de_40,0.00,saldo_cero


,categorias
age_40_plus,2
balance_status,2


Nombres para el siguiente ColumnTransformer: ['geography', 'gender', 'number_of_products', 'credit_score', 'age', 'tenure', 'balance', 'has_credit_card', 'is_active_member', 'estimated_salary', 'age_40_plus', 'balance_status']


## Conclusión

Se proponen dos features categóricas fijas, motivadas por tasas de churn descriptivas de train. Ambas se calculan con columnas disponibles en inferencia, y la clase no aprende la etiqueta ni estadísticas globales. La codificación OneHot se deja para el `ColumnTransformer` del siguiente paso, que se ajustará exclusivamente con cada fold de entrenamiento. Las asociaciones observadas justifican probar estas variables; solo la CV permitirá decidir si realmente aportan.